# 05. Runnable 기본 조합 (LCEL)

- LangChain에서의 Runnable은 입력값을 받아 특정 작업을 수행하고 출력값을 반환하는 실행 가능한 구성요소임.
- 이를 활용하면 프롬프트, 모델, 출력 파서, 사용자 정의 함수, 병렬 처리 흐름을 직관적인 파이프라인으로 표현할 수 있음.
- LCEL(LangChain Expression Language)은 `|` 연산자로 컴포넌트를 체인처럼 연결함.
- Runnable 조합을 사람이 읽기 쉬운 파이프라인 문법으로 표현한 것.


```
prompt | llm | parser
```

In [1]:
import os
import sys
from dotenv import load_dotenv
sys.path.append("..")  # 상위 폴더의 common_config.py를 import하기 위해 추가
from common_config import get_llm

load_dotenv(override=True, dotenv_path="../.env")

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = get_llm("gpt-5.4-mini", temperature=0)

## 1. 기본 체인: prompt | llm | parser

In [2]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}에 대해 한 문장으로 설명해줘")
])

parser = StrOutputParser()

# | 연산자로 체인 구성
chain = prompt | llm | parser

result = chain.invoke({"topic": "벡터 데이터베이스"})
print(result)

벡터 데이터베이스는 텍스트, 이미지 같은 데이터를 의미를 담은 숫자 벡터로 저장하고, 비슷한 의미의 데이터를 빠르게 찾도록 해주는 데이터베이스입니다.


## 2. 체인 스트리밍
- LangChain 체인 실행 결과를 스트리밍 방식으로 조금씩 받아 출력하는 코드

In [3]:
for chunk in chain.stream({"topic": "임베딩"}):
    # 현재 받은 응답 조각을 출력 후 줄바꿈 없이, 버퍼에 쌓아두지 않고 즉시 출력
    print(chunk, end="", flush=True)

임베딩은 **단어, 문장, 이미지 같은 데이터를 컴퓨터가 이해할 수 있도록 의미가 비슷한 것끼리 가까이 배치한 숫자 벡터로 표현하는 방법**입니다.

In [4]:
for chunk in chain.stream({"topic": "임베딩"}):
    # Python이 출력 내용을 잠시 버퍼에 모아두었다가 나중에 보여줌
    print(chunk, end="", flush=False)

임베딩은 텍스트, 이미지 같은 복잡한 데이터를 컴퓨터가 비교하고 계산하기 쉬운 숫자 벡터로 바꾸는 방법입니다.

# 3. Runnable의 공통 실행 메서드
- 입력 값을 받아 어떤 작업을 수행하고, 출력값을 반환하는 표준 실행 단위
- 프롬프트, LLM, 출력파서, Retriever, 사용자 정의 함수 등을 하나의 실행 규칙으로 연결하기 위한 표준 인터페이스
- 기본 : 입력 → Runnable → 출력
- 여러 Runnable 연결 : 입력 → prompt → llm → parser → custom function → 최종 출력 

- Runnable의 공통 실행 메서드 

| 메서드                    | 의미              | 사용 상황           |
| ---------------------- | --------------- | --------------- |
| `invoke()`             | 단일 입력 실행        | 가장 기본 실행        |
| `ainvoke()`            | 비동기 단일 실행       | async 환경        |
| `batch()`              | 여러 입력을 한 번에 실행  | 여러 질문 일괄 처리     |
| `abatch()`             | 비동기 batch 실행    | 대량 처리           |
| `stream()`             | 결과를 조각 단위로 스트리밍 | 챗봇 실시간 출력       |
| `astream()`            | 비동기 스트리밍        | 웹소켓, SSE 등      |
| `batch_as_completed()` | 완료되는 순서대로 결과 반환 | 입력별 처리 시간이 다를 때 |


## 3. RunnablePassthrough - 입력을 그대로 전달
- RunnablePassthrough는 입력값을 거의 그대로 통과시킴

```mermaid
flowchart LR
    subgraph A["RunnablePassthrough()"]
        direction LR
        A1["입력<br/>{'topic': 'RAG'}"] --> A2["RunnablePassthrough"] --> A3["출력<br/>{'topic': 'RAG'}"]
    end

    subgraph B["RunnablePassthrough.assign(upper=...)"]
        direction LR
        B1["입력<br/>{'word': 'hello'}"] --> B2["RunnablePassthrough<br/>(입력 그대로 유지)"]
        B1 --> B3["upper 함수<br/>x['word'].upper()"]
        B2 --> B4["출력<br/>{'word': 'hello',<br/>'upper': 'HELLO'}"]
        B3 --> B4
    end
```

In [5]:
from langchain_core.runnables import RunnablePassthrough

# 입력값을 그대로 흘려보내기
chain = RunnablePassthrough()
result = chain.invoke({"topic": "RAG"})
print(result)

{'topic': 'RAG'}


- `.assign()`은 기존 딕셔너리에 새로운 필드를 추가할 때 사용함

In [6]:
# 실용 예: 입력을 그대로 유지하면서 추가 키 삽입
from langchain_core.runnables import RunnablePassthrough

upper=lambda x: x["word"].upper()
chain = RunnablePassthrough.assign(upper=upper)

result = chain.invoke({"word": "hello"})
print(result)  # {'word': 'hello', 'upper': 'HELLO'}

{'word': 'hello', 'upper': 'HELLO'}


## 4. RunnableLambda - 함수를 Runnable로 감싸기
- RunnableLambda는 일반 파이썬 함수를 Runnable로 감싸는 객체
- RunnableLambda는 체인 중간이나 마지막에 후처리 로직을 넣을 때 주로 사용

In [7]:
from langchain_core.runnables import RunnableLambda

# 문자열 -> 문자열 편집 반환 함수
def add_prefix(text: str) -> str:
    return f"[결과] {text}"

prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}이 뭔가요?")
])

chain = prompt | llm | StrOutputParser() | RunnableLambda(add_prefix)

result = chain.invoke({"topic": "LangGraph"})
print(result)

[결과] LangGraph는 **LLM(대규모 언어모델) 기반 앱을 “그래프 구조”로 설계하고 실행하게 해주는 프레임워크**입니다.  
특히 **복잡한 흐름, 반복, 조건 분기, 상태 유지**가 필요한 AI 에이전트/워크플로우를 만들 때 유용합니다.

### 한마디로
- **LangChain**이 “LLM 도구 모음”에 가깝다면,
- **LangGraph**는 “LLM 실행 흐름을 상태가 있는 그래프로 정의하는 도구”에 가깝습니다.

---

## 왜 필요한가요?
일반적인 LLM 호출은 보통 이렇게 단순합니다:

1. 입력 받기
2. 모델 호출
3. 결과 출력

하지만 실제 에이전트는 더 복잡하죠:

- 사용자의 요청을 이해한 뒤
- 검색이 필요한지 판단하고
- 검색 도구를 쓰고
- 결과를 검토하고
- 부족하면 다시 검색하거나
- 사람의 승인을 기다리기도 하고
- 대화 상태를 유지해야 함

이런 흐름은 **순차 코드만으로는 관리가 어려워서**, LangGraph는 이를 **노드(node)와 엣지(edge)** 로 표현합니다.

---

## 핵심 개념
### 1. 노드(Node)
각 단계의 작업입니다. 예:
- LLM 호출
- 검색 도구 실행
- 메모리 업데이트
- 결과 검증

### 2. 엣지(Edge)
노드 간의 이동 경로입니다.  
조건에 따라 다음 노드가 달라질 수 있습니다.

예:
- “검색이 필요하면” → 검색 노드
- “이미 답변 가능하면” → 답변 노드

### 3. 상태(State)
그래프 전체가 공유하는 데이터입니다.  
예:
- 대화 기록
- 중간 추론 결과
- 검색 결과
- 사용자 설정

LangGraph의 강점은 이 **상태를 유지하면서 여러 번 루프를 돌 수 있다**는 점입니다.

---

## 어디에 많이 쓰이나요?
- **에이전트(Agent)**
- **툴 사용이 필요한 챗봇**
- **다단계 업무 자동화**
- **human-in-the-loop 워크플로우**
- **반복/재시도/검증이 필요한 시스템**

예를 들어:
- 고객 상

## 5. 병렬 실행 - RunnableParallel

딕셔너리 형태로 여러 체인을 동시에 실행합니다.

```mermaid
flowchart LR
    A["입력<br/>{'topic': 'RAG'}"] --> B["chain_ko<br/>한국어 설명"]
    A --> C["chain_en<br/>영어 설명"]
    B --> D["출력<br/>{'korean': ..., 'english': ...}"]
    C --> D
```

In [8]:
from langchain_core.runnables import RunnableParallel

prompt_ko = ChatPromptTemplate.from_messages([("human", "{topic}을 한국어로 설명해줘")])
prompt_en = ChatPromptTemplate.from_messages([("human", "Explain {topic} in English")])

chain_ko = prompt_ko | llm | StrOutputParser()
chain_en = prompt_en | llm | StrOutputParser()

# 두 체인을 병렬로 실행하여 각각 한국어와 영어 설명 생성
parallel_chain = RunnableParallel(
    korean=chain_ko,
    english=chain_en
)

result = parallel_chain.invoke({"topic": "RAG"})
print("[한국어]", result["korean"])
print("[English]", result["english"])

[한국어] RAG는 **Retrieval-Augmented Generation**의 약자로, 한국어로는 보통 **검색 증강 생성**이라고 합니다.

쉽게 말하면:

- **검색(Retrieval)**: 필요한 정보를 외부 문서, 데이터베이스, 웹 등에서 찾아오고
- **증강(Augmented)**: 그 찾은 정보를
- **생성(Generation)**: AI가 답변 생성할 때 함께 활용하는 방식입니다.

## 왜 쓰나?
대형 언어모델은 학습된 지식만으로 답하므로:
- 최신 정보에 약할 수 있고
- 특정 회사 문서, 내부 규정, 매뉴얼 같은 전용 정보는 모를 수 있습니다.

RAG는 이런 문제를 줄이기 위해,
**질문에 맞는 관련 자료를 먼저 찾아오고, 그 자료를 참고해서 답변**하게 합니다.

## 예시
사용자가  
“우리 회사의 연차 휴가 규정이 뭐야?”라고 물으면,

1. 시스템이 사내 인사 규정 문서를 검색하고
2. 관련 부분을 찾아
3. 그 내용을 바탕으로 AI가 답변합니다.

## 장점
- 최신 정보 반영 가능
- 내부 문서 기반 답변 가능
- 환각(hallucination) 줄이는 데 도움
- 출처를 함께 제시하기 좋음

## 한 줄 요약
**RAG는 AI가 답을 그냥 만들어내는 게 아니라, 먼저 관련 정보를 찾아보고 그걸 바탕으로 답하게 하는 방식**입니다.

원하시면 제가 **RAG 구조를 그림처럼 단계별로** 더 쉽게 설명해드릴게요.
[English] RAG stands for **Retrieval-Augmented Generation**.

In plain English, it means:

1. **Retrieve** relevant information from a source, like documents, a database, or the web.
2. **Augment** the model’s input with that information.
3. **Generate** an answer using both the retrieved 

## 실습 문제
Q2. RunnableLambda를 활용해 LLM 응답을 받아 단어 수를 세는 함수를 파이프라인 끝에 추가해보세요. 결과로 {'text': '...', 'word_count': 42} 형태로 반환하세요.

In [10]:
from langchain_core.runnables import RunnableLambda

# 1. 프롬프트 구성
prompt = ChatPromptTemplate.from_template(
    "{topic}에 대해 초보자도 이해할 수 있게 설명해줘."
)

# 2. LLM 생성
llm = get_llm("gpt-5.4-mini", temperature=0)

# 3. 출력 파서 생성
parser = StrOutputParser()

# 4. LLM 응답을 받아 단어 수를 세는 함수
def count_words(text: str) -> dict:
    return {
        "text": text,
        "word_count": len(text.split())
    }

# 5. RunnableLambda로 함수 감싸기
word_count_runnable = RunnableLambda(count_words)

# 6. LCEL 체인 구성
chain = prompt | llm | parser | word_count_runnable

# 7. 실행
result = chain.invoke({"topic": "LangChain Runnable"})

print(result["word_count"])  # 설명의 단어 수
print("-"*20)
print(result["text"])        # LLM이 생성한 설명

460
--------------------
물론이죠.  
**LangChain의 Runnable**을 아주 쉽게 말하면:

> **“여러 작업을 한 줄로 이어 붙여 실행할 수 있게 해주는 표준 실행 단위”** 입니다.

---

## 1) Runnable이 왜 필요한가?

LangChain에서는 보통 이런 작업을 합니다.

- 사용자 질문 받기
- 프롬프트 만들기
- LLM에 보내기
- 결과 후처리하기
- 다른 도구 실행하기

이런 것들을 각각 따로 만들 수도 있지만, LangChain은 이 과정을 **연결해서 파이프라인처럼 처리**하고 싶습니다.

그때 쓰는 핵심 개념이 **Runnable**입니다.

---

## 2) 아주 쉬운 비유

Runnable은 **레고 블록**처럼 생각하면 됩니다.

- 각 블록은 하나의 작업을 함
- 블록들을 순서대로 연결할 수 있음
- 입력이 들어오면 연결된 순서대로 처리됨

예를 들어:

1. 문장을 받음
2. 영어로 번역함
3. 요약함
4. 결과를 출력함

이렇게 이어 붙인 각 단계가 Runnable이 될 수 있습니다.

---

## 3) Runnable이 하는 일

Runnable은 기본적으로 다음과 같은 역할을 합니다.

- **입력(input)** 을 받음
- **출력(output)** 을 반환함
- 다른 Runnable과 **합성(composition)** 가능함
- **체인(chain)** 을 쉽게 만들 수 있음

즉, “무언가를 실행할 수 있는 객체”라고 보면 됩니다.

---

## 4) 가장 기본적인 예

LangChain에서는 Runnable을 `|` 연산자로 연결할 수 있습니다.

예:

```python
chain = prompt | llm | parser
```

이 뜻은:

- `prompt`가 입력을 받아 프롬프트를 만들고
- `llm`이 그 프롬프트를 처리하고
- `parser`가 결과를 원하는 형태로 바꾼다

즉, **왼쪽 결과가 오른쪽 입력으로 들어가는 구조**입니다.

---

## 5